# PolicyGuard-4B: 100 Test Cases Inference

Simple, single-purpose inference pipeline:
1. Loads the pretrained `Rakancorle1/PolicyGuard-4B` model on GPU.
2. Reads the 100 test case `.txt` files from `policyguard_prompts_100/policyguard_inputs`.
3. Sends each prompt to the model and collects the exact raw output.
4. Saves all outputs mapped to test case IDs directly to `policyguard_outputs.csv`.


## Cell 1: Environment Setup

> **Runtime Setup:**
> - **Google Colab:** `Runtime -> Change runtime type -> Hardware accelerator: T4 GPU`.
> - **Kaggle Notebook:** Right-side Settings panel -> `Accelerator: GPU T4 x2` **AND** turn `Internet: ON`.


In [ ]:
import torch

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA is available! GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: CUDA is NOT available. Please select a GPU runtime (T4 GPU).")

# Install minimal required packages
!pip install -q transformers>=4.40.0 accelerate>=0.28.0 safetensors>=0.4.0 pandas


## Cell 2: Repository Setup

Clones or navigates into the repository and imports the inference functions.


In [ ]:
import os
import sys
import importlib

# Clone or navigate into GuardSLM repository
if not os.path.exists("policyguard_pipeline.py"):
    if not os.path.exists("GuardSLM"):
        os.system("git clone https://github.com/AkarshiAaryan/GuardSLM.git")
    if os.path.exists("GuardSLM"):
        os.chdir("GuardSLM")
elif os.path.isdir("GuardSLM") and os.path.basename(os.getcwd()) != "GuardSLM":
    os.chdir("GuardSLM")

if "." not in sys.path:
    sys.path.insert(0, ".")

# Pull latest changes
if os.path.exists(".git"):
    os.system("git pull origin main")

# Force reload in active kernel
import policyguard_pipeline
importlib.reload(policyguard_pipeline)

from policyguard_pipeline import load_model, run_inference
print("Successfully imported policyguard_pipeline.")


## Cell 3: Load PolicyGuard-4B Model

Loads the pretrained tokenizer and model once into GPU memory.


In [ ]:
tokenizer, model = load_model(model_id="Rakancorle1/PolicyGuard-4B")
print("PolicyGuard-4B model loaded successfully.")


## Cell 4: Run Inference on All 100 Test Cases

Sends each test case prompt to the model and saves all raw outputs to `policyguard_outputs.csv`.


In [ ]:
INPUT_DIR = "policyguard_prompts_100/policyguard_inputs"
OUTPUT_CSV = "policyguard_outputs.csv"

results = run_inference(
    input_dir=INPUT_DIR,
    output_csv=OUTPUT_CSV,
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=4
)


## Cell 5: Display Collected Outputs Table

Loads and displays `policyguard_outputs.csv`.


In [ ]:
import pandas as pd
from IPython.display import display

df = pd.read_csv("policyguard_outputs.csv")
print(f"Total Test Cases Processed: {len(df)}")
print("Saved Output File: policyguard_outputs.csv\n")

display(df)


## Cell 6: Cleanup

Safely releases model memory from GPU.


In [ ]:
import gc

print("Releasing model and clearing GPU cache...")
del model
del tokenizer
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU cache cleared.")
print("Cleanup complete.")
